# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps my Week 1 lane onto the ML loop. Sections follow the skeleton in order:
task type → target → metric → real dataframe of the unit of analysis → why ML and not a rule.
Every claim uses careful words: observed, measured, directional, decision-support.

## 1. My lane as an ML task (type)

**Ranking / scoring.** Lane 2 from Week 1 is Refresh / Content Opportunity Scoring: a FlyRank
editor has a fixed review budget (a handful of pages per session), a client's library holds
thousands of pages, and something has to decide the **order** in which those pages get looked at.

- I output a **continuous priority score** per page, and the editor works down the top of the
  sorted queue. That is scoring, evaluated as a ranking.
- It is **not classification**: nobody wants a yes/no verdict stamped on all 30,000 pages. The
  scarce resource — editor attention — is consumed in ranked order, so "is the top of the queue
  right?" is the objective, not "is every page labelled correctly?".
- It is **not clustering**: grouping lookalike pages is not the decision; ordering them for
  review is.

**The frame in one paragraph.** For FlyRank content editors, deciding *which pages in a client's
library to review first this week*, we will build a **priority score** from trailing-90-day
search and engagement metrics, predicting **"page is in decline"** measured by **precision@K on
the top of the queue**. A wrong call costs editor hours (false positive) or a silently declining
page (false negative). A plain rule is not enough because the causes of decline sit in the
*combination* of many weak signals (evidence in section 5). We will claim only observed /
directional / decision-support results.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

def find_raw_csv():
    """Locate the starter CSV wherever this notebook is run from (repo root or notebooks/)."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "data" / "raw" / "content_refresh_anonymized.csv"
        if cand.exists():
            return cand
    return None

RAW = find_raw_csv()
assert RAW is not None, "starter CSV not found under data/raw/"
df = pd.read_csv(RAW)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(f"starter slice: {len(df):,} rows x {len(df.columns)} cols, {df['client_id'].nunique()} clients")
print(f"reviewable pool (impressions_90d >= 100 AND sessions_90d > 0): "
      f"{((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).sum():,} of {len(df):,} pages"
      f" ({100 * ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).mean():.0f}%)")
print(f"-> an editor cannot hand-triage tens of thousands of pages; the decision is a ranking.")

starter slice: 30,000 rows x 45 cols, 32 clients
reviewable pool (impressions_90d >= 100 AND sessions_90d > 0): 22,006 of 30,000 pages (73%)
-> an editor cannot hand-triage tens of thousands of pages; the decision is a ranking.


## 2. Target or proxy

**What I rank by = the predicted probability of the label "this page is in decline".**

- **Honest target (warehouse, weeks 3+):** an *observed* binary outcome measured in a **later**
  time window — `declined_next_30d = 1` when forwarding-month impressions on the daily fact
  table fall materially vs the preceding 30 days. Features come from the **prior** 90 days. This
  is observed, not defined by a rule.
- **The only label the starter slice ships is a PROXY:** `is_declining_label =
  (trend_direction == "down")`, and `trend_direction` is itself computed from `trend_pct`
  — it lives in the **same window** the features live in. It is a *defined rule*, not an
  observed outcome, so I can use it here only while saying exactly what it is, and I will never
  claim it as a causal outcome.

The code below (a) proves the proxy is literally that rule, (b) shows its distribution, and
(c) sketches what the honest warehouse target column would look like (not computable on this
slice, so shown as a placeholder).

In [2]:
parity = (df["is_declining_label"] == (df["trend_direction"].str.lower() == "down")).all()
print("is_declining_label is exactly (trend_direction == 'down')?", parity)

print("\ntrend_direction distribution (never a feature — it IS the label source):")
print(df["trend_direction"].value_counts().sort_index().to_string())

print("\nproxy label distribution on the starter slice:")
print(df["is_declining_label"].value_counts().sort_index().to_string())
print(f"base rate: {df['is_declining_label'].mean():.3f}")

# Sketch of the honest target column — observed next-30-day outcome, only measurable on the
# warehouse's daily fact table (future rows). Here it is a placeholder on purpose.
sketch = df.head(5).copy()
sketch["declined_next_30d"] = pd.NA
show_cols = ["content_id", "content_type", "trend_direction", "is_declining_label", "declined_next_30d"]
print("\nsketch — what the target column looks like (placeholder on starter; observed on warehouse):")
sketch[show_cols]

is_declining_label is exactly (trend_direction == 'down')? True

trend_direction distribution (never a feature — it IS the label source):
trend_direction
down      16262
flat       1152
new        2236
stable     5962
up         4388

proxy label distribution on the starter slice:
is_declining_label
0    13738
1    16262
base rate: 0.542

sketch — what the target column looks like (placeholder on starter; observed on warehouse):


,content_id,content_type,trend_direction,is_declining_label,declined_next_30d
0,content_304f48230142,keyword article,down,1,<NA>
1,content_a1fb4e703a9e,keyword article,down,1,<NA>
2,content_9aa793d4d895,keyword article,down,1,<NA>
3,content_331d6c4de07b,keyword article,stable,0,<NA>
4,content_d99b7a2d90ca,keyword article,down,1,<NA>


## 3. Success metric

**One defendable number: precision@K on the top of the ranked queue — primary K = 50** (a
reviewing session's plausible workload; the starter pipeline also tracks K = 20).

precision@K = share of the top-K review picks whose label is 1.

- The base rate is 54.2%, so "random" already scores ~0.54 at any K. "Good" must mean **beating
  chance and beating the hand-written rule** at the exact point the queue is consumed — the top.
- Accuracy and overall AUC are the wrong headline here: they reward getting ~29,950 low-priority
  rows right, none of which an editor will ever open.
- Concretely, on the committed starter evaluation (`outputs/model_report.md`, client-holdout,
  same proxy label): random_forest reached precision@50 = **0.740** vs **0.240** for the
  rule-based score. That gap is the "good" bar to re-test once the warehouse gives a genuine
  forward-window label.

The table below shows that obvious single-signal rules do not even clear chance — which is why
the metric has to be measured against a learned combination, not assumed to be beaten by one.

In [3]:
label = "is_declining_label"

rule_p50 = pd.DataFrame({
    "rule (top-50 pick order)": [
        "random pick (base rate)",
        "highest impressions_90d",
        "oldest (days_since_last_update)",
        "lowest ctr",
        "best avg_position (excluding the 1,205 no-data rows)",
    ],
    "precision@50": [
        df[label].mean(),
        df.sort_values("impressions_90d", ascending=False).head(50)[label].mean(),
        df.sort_values("days_since_last_update", ascending=False).head(50)[label].mean(),
        df.sort_values("ctr", ascending=True).head(50)[label].mean(),
        df[df["avg_position"] > 0].sort_values("avg_position").head(50)[label].mean(),
    ],
})

print("Committed client-holdout evaluation (outputs/model_report.md, same label):")
print("  baseline_rules  precision@50 = 0.240")
print("  random_forest   precision@50 = 0.740  (best of the trained models)")
print("\n'Good' here = clearly above 0.542 (chance) and above the 0.240 hand-rule, at K = 50.")
rule_p50.round(3)

Committed client-holdout evaluation (outputs/model_report.md, same label):
  baseline_rules  precision@50 = 0.240
  random_forest   precision@50 = 0.740  (best of the trained models)

'Good' here = clearly above 0.542 (chance) and above the 0.240 hand-rule, at K = 50.


,rule (top-50 pick order),precision@50
0,random pick (base rate),0.542
1,highest impressions_90d,0.420
2,oldest (days_since_last_update),0.520
3,lowest ctr,0.540
4,"best avg_position (excluding the 1,205 no-data...",0.160


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page).** 30,000 rows, 30,000 unique `content_id`s, across 32
pseudonymized clients. Each row is a page-level snapshot: content properties + trailing-90-day
search/engagement totals + derived rates + the label. (On the warehouse the *same* pages expand
to many daily rows in `fact_content_daily_performance`, but the decision unit stays the page —
that is precisely what lands in an editor's queue.)

The dataframe below is the unit of analysis: identity (pseudonym), page metadata, the core
signals the ranking consumes, and the label. Grain is proven first: no page repeats.

In [4]:
print(f"rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} "
      f"| page-per-row true: {df['content_id'].nunique() == len(df)}"
      f" | clients: {df['client_id'].nunique()}")

cols = ["content_id", "client_id", "content_type", "content_age_days",
        "days_since_last_update", "impressions_90d", "avg_position", "ctr",
        "trend_direction", "is_declining_label"]
df[cols].head(8)

rows: 30,000 | unique content_id: 30,000 | page-per-row true: True | clients: 32


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,21.2,0.06,stable,0


## 5. Why ML beats a fixed rule here

Three reasons, with numbers below:

1. **No single signal carries the label.** Every individual feature correlates with the proxy
   label at |r| ≤ ~0.19 on this slice. A rule must read several weak signals *at once*; the
   single-threshold rules in section 3 all land at or below chance (0.42–0.54).
2. **The right combination shifts by strata.** A threshold that works in one
   `content_type x competition_level x impression_tier` cell misbehaves in another, and client
   traffic scales differ by orders of magnitude. A hand-rule becomes a pile of hand-tuned
   numbers that quietly break when the client mix changes.
3. **It is already demonstrable on this data.** On a client-holdout, a model combining ~15
   signals scores precision@50 = 0.740 vs 0.240 for the hand-written rule — the learned
   combination finds declining pages the rule missed, at the top of the queue where it matters.

That matches the framing skill's test for whether ML earns its place: the pattern is real but
too messy to write by hand — and here it is testable, not assumed.

In [5]:
feats = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
         "days_since_last_update", "content_age_days", "word_count",
         "days_with_impressions", "engagement_rate"]
corr = pd.Series({f: df[f].corr(df["is_declining_label"]) for f in feats})

cells = df.groupby(["content_type", "competition_level", "impression_tier"], dropna=False).ngroups
print(f"distinct content_type x competition_level x impression_tier cells on this slice: {cells}")
print("-> a hand-written rule needs thresholds per strata like these (22 already on this small slice).")
print("\nsingle-signal top-50 rules: 0.42 / 0.52 / 0.54 / 0.54  (base rate 0.542)")
print("combined ~15-signal model (client-holdout): 0.740  -> ML beats the rule where it counts.")
corr.round(3).sort_values(key=lambda s: s.abs(), ascending=False)

distinct content_type x competition_level x impression_tier cells on this slice: 22
-> a hand-written rule needs thresholds per strata like these (22 already on this small slice).

single-signal top-50 rules: 0.42 / 0.52 / 0.54 / 0.54  (base rate 0.542)
combined ~15-signal model (client-holdout): 0.740  -> ML beats the rule where it counts.


days_with_impressions     0.190
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
clicks_90d               -0.040
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
dtype: float64

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.